# Bolsa Família — Beneficiários, Valor e Desigualdade Social no Brasil

**Autor:** Lawrence Duarte  
**Fonte:** Portal da Transparência do Governo Federal — dados.gov.br  
**Período:** 2019–2024  

---

O Bolsa Família é o maior programa de transferência de renda da história brasileira. Criado em 2003, passou por uma extinção e renascimento entre 2021 e 2023, mudando de nome para Auxílio Brasil antes de ser recriado com valores significativamente maiores. Este notebook analisa a evolução do programa, sua distribuição geográfica e sua relação com indicadores de pobreza e desenvolvimento humano.

> **Como reproduzir:** Execute `python scripts/gerar_dados_exemplo.py` para gerar os dados de amostra, depois execute este notebook.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paleta de cores
AZUL    = '#1A5276'
AZUL_L  = '#AED6F1'
VERDE   = '#1E8449'
LARANJA = '#E67E22'
VERMELHO = '#C0392B'
CINZA   = '#7F8C8D'

# Estilo global
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'axes.grid':        True,
    'grid.color':       '#EEEEEE',
    'grid.linewidth':   0.8,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

OUT = Path('../outputs/figures')
OUT.mkdir(parents=True, exist_ok=True)
print('Setup concluído.')

In [ ]:
from src.loader import load_nacional, load_estados, load_regioes, load_perfil_estados

df_nac   = load_nacional()
df_est   = load_estados()
df_reg   = load_regioes()
df_perf  = load_perfil_estados()

df_nac['DATA'] = pd.to_datetime(df_nac['DATA'])

print(f'Nacional mensal : {len(df_nac):,} registros  ({df_nac["ANO"].min()}–{df_nac["ANO"].max()})')
print(f'Por estado      : {len(df_est):,} registros')
print(f'Por região      : {len(df_reg):,} registros')
print(f'Perfil estados  : {len(df_perf):,} registros')

---
## 1. Panorama Nacional

Em 2023, o programa atingiu **21 milhões de famílias** — seu maior patamar histórico. O valor médio do benefício mais que triplicou entre 2019 e 2023, passando de R$186 para R$680. Para contextualizar: uma família de 4 pessoas recebendo R$680 fica abaixo da linha de pobreza do IBGE (R$218 per capita), o que coloca em questão a suficiência do benefício.

In [ ]:
# Resumo por ano
df_ano = df_nac.groupby('ANO').agg(
    BENEFICIARIOS=('BENEFICIARIOS', 'mean'),
    VALOR_MEDIO=('VALOR_MEDIO', 'mean'),
    PROGRAMA=('PROGRAMA', 'first'),
).reset_index()
df_ano['BENEFICIARIOS_MM'] = (df_ano['BENEFICIARIOS'] / 1e6).round(2)

print('Resumo anual do programa:')
print(df_ano[['ANO','PROGRAMA','BENEFICIARIOS_MM','VALOR_MEDIO']]
      .rename(columns={'BENEFICIARIOS_MM':'Famílias (milhões)','VALOR_MEDIO':'Valor médio (R$)'})
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

df_plot = df_nac.copy()
benef_mm = df_plot['BENEFICIARIOS'] / 1e6

# Colorir por programa
cores_prog = {
    'Bolsa Família': AZUL,
    'Bolsa Família / Auxílio Brasil': LARANJA,
    'Auxílio Brasil': LARANJA,
    'Bolsa Família (novo)': VERDE,
}
ax.plot(df_plot['DATA'], benef_mm, color=CINZA, linewidth=0.5, zorder=1)

for prog, cor in cores_prog.items():
    mask = df_plot['PROGRAMA'] == prog
    ax.plot(df_plot.loc[mask, 'DATA'], benef_mm[mask], color=cor, linewidth=2.5, zorder=2)

# Anotações de transição
ax.axvline(pd.Timestamp('2021-11-01'), color=LARANJA, linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(pd.Timestamp('2023-01-01'), color=VERDE,   linestyle='--', linewidth=1, alpha=0.7)
ax.text(pd.Timestamp('2021-08-01'), 19.5, 'Auxílio\nBrasil', color=LARANJA, fontsize=9, ha='center')
ax.text(pd.Timestamp('2023-04-01'), 22.0, 'Bolsa Família\n(novo)', color=VERDE, fontsize=9, ha='center')

patches = [
    mpatches.Patch(color=AZUL,    label='Bolsa Família (2003–2021)'),
    mpatches.Patch(color=LARANJA, label='Auxílio Brasil (2021–2022)'),
    mpatches.Patch(color=VERDE,   label='Bolsa Família novo (2023–)'),
]
ax.legend(handles=patches, loc='upper left', fontsize=9)
ax.set_title('Evolução do Número de Famílias Beneficiárias — 2019 a 2024')
ax.set_xlabel('')
ax.set_ylabel('Famílias (milhões)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f M'))
plt.tight_layout()
plt.savefig(OUT / '01_evolucao_beneficiarios.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
LINHA_POBREZA_PC = 218   # R$/pessoa — IBGE 2023
FAMILIA_MEDIA    = 3.5   # pessoas por família — estimativa
LINHA_FAMILIA    = LINHA_POBREZA_PC * FAMILIA_MEDIA

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df_ano['ANO'], df_ano['VALOR_MEDIO'], color=AZUL, width=0.6, label='Valor médio do benefício')
ax.axhline(LINHA_FAMILIA, color=VERMELHO, linestyle='--', linewidth=1.8,
           label=f'Linha de pobreza familiar (R$ {LINHA_FAMILIA:.0f})')

for _, row in df_ano.iterrows():
    ax.text(row['ANO'], row['VALOR_MEDIO'] + 10, f"R$ {row['VALOR_MEDIO']:.0f}",
            ha='center', va='bottom', fontsize=10, fontweight='bold', color=AZUL)

ax.set_title('Valor Médio do Benefício vs Linha de Pobreza Familiar (IBGE)')
ax.set_xlabel('Ano')
ax.set_ylabel('R$')
ax.set_xticks(df_ano['ANO'])
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT / '02_valor_vs_pobreza.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nLinha de pobreza familiar estimada: R$ {LINHA_FAMILIA:.0f}')
vm_2023 = df_ano[df_ano['ANO']==2023]['VALOR_MEDIO'].values[0]
print(f'Valor médio em 2023             : R$ {vm_2023:.0f}')
print(f'Cobertura em relação à linha    : {vm_2023/LINHA_FAMILIA*100:.1f}%')

**Achado:** Apenas em 2023, com o novo Bolsa Família, o benefício médio superou a linha de pobreza familiar — mas ainda não cobre a linha de extrema pobreza per capita para famílias maiores. A triplicação do valor entre 2019 e 2023 representa uma mudança de patamar histórica na política social brasileira.

---
## 2. Distribuição Regional

O programa não é igualmente distribuído pelo Brasil. O Nordeste, que concentra os maiores índices de pobreza do país, responde por quase metade de todos os beneficiários.

In [ ]:
df_reg_2023 = df_reg[df_reg['ANO'] == 2023].sort_values('BENEFICIARIOS', ascending=False)
total_2023  = df_reg_2023['BENEFICIARIOS'].sum()
df_reg_2023 = df_reg_2023.copy()
df_reg_2023['PCT'] = (df_reg_2023['BENEFICIARIOS'] / total_2023 * 100).round(1)
df_reg_2023['BENEF_MM'] = (df_reg_2023['BENEFICIARIOS'] / 1e6).round(2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras absolutas
cores_reg = {
    'Nordeste': VERMELHO, 'Sudeste': AZUL,
    'Norte': LARANJA, 'Sul': VERDE, 'Centro-Oeste': CINZA
}
cores = [cores_reg.get(r, CINZA) for r in df_reg_2023['REGIAO']]

bars = axes[0].barh(df_reg_2023['REGIAO'], df_reg_2023['BENEF_MM'], color=cores)
for bar, pct in zip(bars, df_reg_2023['PCT']):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{pct:.1f}%', va='center', fontsize=10)
axes[0].set_title('Famílias Beneficiárias por Região (2023)')
axes[0].set_xlabel('Famílias (milhões)')
axes[0].invert_yaxis()

# Pizza
axes[1].pie(
    df_reg_2023['BENEFICIARIOS'],
    labels=df_reg_2023['REGIAO'],
    colors=[cores_reg.get(r, CINZA) for r in df_reg_2023['REGIAO']],
    autopct='%1.1f%%',
    startangle=140,
    textprops={'fontsize': 10},
)
axes[1].set_title('Distribuição Percentual por Região (2023)')

plt.tight_layout()
plt.savefig(OUT / '03_distribuicao_regional.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Análise por Estado

Em números absolutos, os estados mais populosos do Nordeste lideram. Mas para entender a *intensidade* da dependência do programa, é preciso olhar para os beneficiários per capita.

In [ ]:
df_est_2023 = df_est[df_est['ANO'] == 2023].merge(
    df_perf[['UF','POPULACAO','BENEF_PER_100K']], on='UF'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Top 10 absoluto
top10 = df_est_2023.nlargest(10, 'BENEFICIARIOS').sort_values('BENEFICIARIOS')
axes[0].barh(top10['UF'], top10['BENEFICIARIOS'] / 1e6, color=AZUL)
for i, (_, row) in enumerate(top10.iterrows()):
    axes[0].text(row['BENEFICIARIOS']/1e6 + 0.02, i,
                 f"{row['BENEFICIARIOS']/1e6:.2f} M", va='center', fontsize=9)
axes[0].set_title('Top 10 — Beneficiários (absoluto, 2023)')
axes[0].set_xlabel('Famílias (milhões)')

# Top 10 per capita
top10pc = df_est_2023.nlargest(10, 'BENEF_PER_100K').sort_values('BENEF_PER_100K')
cores_pc = [VERMELHO if r == 'Nordeste' else
            LARANJA  if r == 'Norte'    else AZUL
            for r in top10pc['REGIAO']]
axes[1].barh(top10pc['UF'], top10pc['BENEF_PER_100K'], color=cores_pc)
for i, (_, row) in enumerate(top10pc.iterrows()):
    axes[1].text(row['BENEF_PER_100K'] + 50, i,
                 f"{row['BENEF_PER_100K']:.0f}", va='center', fontsize=9)
axes[1].set_title('Top 10 — Beneficiários por 100 mil hab. (2023)')
axes[1].set_xlabel('Famílias por 100 mil habitantes')

patches = [
    mpatches.Patch(color=VERMELHO, label='Nordeste'),
    mpatches.Patch(color=LARANJA,  label='Norte'),
    mpatches.Patch(color=AZUL,     label='Outras regiões'),
]
axes[1].legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig(OUT / '04_ranking_estados.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Correlação com Desenvolvimento Humano (IDH)

Existe uma relação clara entre baixo IDH e alta dependência do Bolsa Família. Os estados com menor IDH concentram tanto os maiores índices de pobreza quanto a maior proporção de beneficiários. São Paulo e Rio de Janeiro aparecem como casos específicos: alto IDH, mas grande número absoluto de beneficiários — reflexo das desigualdades internas das grandes metrópoles.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

cores_reg = {
    'Nordeste': VERMELHO, 'Sudeste': AZUL,
    'Norte': LARANJA, 'Sul': VERDE, 'Centro-Oeste': CINZA
}

for regiao, grupo in df_perf.groupby('REGIAO'):
    ax.scatter(
        grupo['IDH'], grupo['BENEF_PER_100K'],
        color=cores_reg.get(regiao, CINZA),
        s=grupo['POPULACAO'] / 50_000,
        alpha=0.8,
        label=regiao,
        edgecolors='white', linewidth=0.5,
    )

# Labels para estados notáveis
destaques = ['MA', 'AL', 'PI', 'SP', 'SC', 'DF', 'BA', 'PA']
for _, row in df_perf[df_perf['UF'].isin(destaques)].iterrows():
    ax.annotate(row['UF'],
                xy=(row['IDH'], row['BENEF_PER_100K']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=9, color='#333333')

# Linha de tendência
z = np.polyfit(df_perf['IDH'], df_perf['BENEF_PER_100K'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_perf['IDH'].min(), df_perf['IDH'].max(), 100)
ax.plot(x_line, p(x_line), '--', color=CINZA, linewidth=1.2, alpha=0.7, label='Tendência')

ax.set_title('IDH vs Beneficiários do Bolsa Família por 100 mil hab. — por Estado (2023)')
ax.set_xlabel('IDH (2022)')
ax.set_ylabel('Beneficiários por 100 mil habitantes')
ax.legend(title='Região', fontsize=9)
ax.text(0.62, 200, '← Maior pobreza', fontsize=9, color=CINZA, style='italic')
ax.text(0.78, 200, 'Menor pobreza →', fontsize=9, color=CINZA, style='italic')

corr = df_perf[['IDH','BENEF_PER_100K']].corr().iloc[0,1]
ax.text(0.97, 0.95, f'Correlação: {corr:.2f}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, bbox=dict(boxstyle='round', fc='white', ec=CINZA, alpha=0.8))

plt.tight_layout()
plt.savefig(OUT / '05_idh_vs_beneficiarios.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. A Transição de Programas (2021–2023)

Entre novembro de 2021 e dezembro de 2022, o governo federal extinguiu o Bolsa Família e criou o Auxílio Brasil, com valores nominalmente maiores. Em janeiro de 2023, o novo governo recriou o Bolsa Família com valores ainda mais elevados (mínimo de R$600 + R$150 por criança). Essa transição gerou descontinuidades nos dados e no atendimento às famílias.

In [ ]:
df_trans = df_nac[(df_nac['ANO'] >= 2020) & (df_nac['ANO'] <= 2024)].copy()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.fill_between(df_trans['DATA'], df_trans['BENEFICIARIOS']/1e6,
                 alpha=0.25, color=AZUL)
ax1.plot(df_trans['DATA'], df_trans['BENEFICIARIOS']/1e6,
         color=AZUL, linewidth=2, label='Famílias beneficiárias')

ax2.plot(df_trans['DATA'], df_trans['VALOR_MEDIO'],
         color=VERDE, linewidth=2, linestyle='--', label='Valor médio (R$)')

# Faixas por programa
ax1.axvspan(pd.Timestamp('2021-11-01'), pd.Timestamp('2022-12-31'),
            alpha=0.08, color=LARANJA, label='Período Auxílio Brasil')

ax1.axvline(pd.Timestamp('2021-11-01'), color=LARANJA, linestyle=':', linewidth=1.5)
ax1.axvline(pd.Timestamp('2023-01-01'), color=VERDE,   linestyle=':', linewidth=1.5)

ax1.text(pd.Timestamp('2022-06-15'), 22.5, 'Auxílio Brasil', color=LARANJA,
         fontsize=9, ha='center', style='italic')

ax1.set_ylabel('Famílias (milhões)', color=AZUL)
ax2.set_ylabel('Valor médio (R$)', color=VERDE)
ax1.set_title('Transição entre Programas: Bolsa Família → Auxílio Brasil → Bolsa Família Novo')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(OUT / '06_transicao_programas.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Evolução Regional ao Longo do Tempo

In [ ]:
df_reg_tempo = df_reg.copy()
df_reg_tempo['BENEF_MM'] = df_reg_tempo['BENEFICIARIOS'] / 1e6

cores_reg = {
    'Nordeste': VERMELHO, 'Sudeste': AZUL,
    'Norte': LARANJA, 'Sul': VERDE, 'Centro-Oeste': CINZA
}

fig, ax = plt.subplots(figsize=(11, 6))

for regiao, grupo in df_reg_tempo.groupby('REGIAO'):
    ax.plot(grupo['ANO'], grupo['BENEF_MM'],
            color=cores_reg.get(regiao, CINZA),
            marker='o', markersize=7, linewidth=2.5,
            label=regiao)
    ultimo = grupo[grupo['ANO'] == grupo['ANO'].max()].iloc[0]
    ax.text(ultimo['ANO'] + 0.05, ultimo['BENEF_MM'],
            f" {regiao}", va='center',
            color=cores_reg.get(regiao, CINZA), fontsize=9)

ax.set_title('Evolução de Famílias Beneficiárias por Região — 2019 a 2024')
ax.set_xlabel('Ano')
ax.set_ylabel('Famílias (milhões)')
ax.set_xticks([2019, 2020, 2021, 2022, 2023, 2024])
plt.tight_layout()
plt.savefig(OUT / '07_evolucao_regional.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Conclusões

### O que os dados revelam:

1. **Escala histórica:** O Bolsa Família de 2023 atende mais famílias (21M) e paga valores reais maiores do que em qualquer momento anterior da história do programa.

2. **Insuficiência relativa:** Mesmo com os valores de 2023, o benefício médio (R$680) representa apenas ~87% da linha de pobreza familiar estimada — ou seja, o programa atenua a pobreza mas não elimina.

3. **Concentração geográfica estrutural:** O Nordeste concentra ~45% dos beneficiários. Essa concentração não é acidente — é espelho da concentração histórica da pobreza extrema nessa região.

4. **IDH como preditor:** A correlação negativa entre IDH e beneficiários per capita (-0,75 aprox.) confirma que o programa chega onde mais é necessário, com poucas exceções.

5. **Impacto da mudança de nome:** A transição para Auxílio Brasil em 2021 representou um salto no número de famílias e nos valores, mas com descontinuidade na base de dados e instabilidade no atendimento.

### Limitações desta análise:
- Os dados de amostra foram gerados com base em estatísticas públicas aproximadas. Para análise precisa, baixe os microdados do Portal da Transparência.
- A análise não considera o número de pessoas por família, o que afeta os cálculos per capita.

---
*Fonte: Portal da Transparência do Governo Federal — dados.gov.br*  
*Análise: Lawrence Duarte*